<a href="https://colab.research.google.com/github/vandan-tarde/Model-1/blob/main/Fault_Detection_in_Transmission_Line5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.1 MB/s eta 0:00:00


In [2]:
# Step 1: Upload and Load Dataset
from google.colab import files
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report



In [5]:
from google.colab import files
uploaded = files.upload()

Saving classData.csv to classData.csv
Saving detect_dataset.csv to detect_dataset.csv


In [6]:
# Load datasets
detect_dataset = pd.read_csv('detect_dataset.csv')
class_data = pd.read_csv('classData.csv')

In [7]:
# Combine datasets
detect_dataset['G'] = 0
detect_dataset['C'] = 0
detect_dataset['B'] = 0
detect_dataset['A'] = detect_dataset['Output (S)']
combined_data = pd.concat([detect_dataset, class_data], ignore_index=True)


In [8]:
# Drop unnecessary columns (for single-phase system)
combined_data.drop(columns=['Unnamed: 7', 'Unnamed: 8', 'Ib', 'Ic', 'Vb', 'Vc'], inplace=True)

# Fill missing values
combined_data.fillna(0, inplace=True)


In [9]:
# Step 2: Fault Distance Calculation for Single Phase
Z_line = 0.021  # Impedance per unit length (Ohms per meter)

def calculate_fault_distance(row):
    return abs(row['Va']) / (abs(row['Ia']) * Z_line) if row['Ia'] != 0 else 0

combined_data['fault_distance'] = combined_data.apply(calculate_fault_distance, axis=1)
combined_data['fault_distance'] = combined_data['fault_distance'].clip(lower=0, upper=7)


In [10]:
# Step 3: Normalize Features
features = ['Ia', 'Va', 'fault_distance']
scaler = StandardScaler()
combined_data[features] = scaler.fit_transform(combined_data[features])


In [11]:
# Step 4: Prepare Data for Training
X = combined_data[features]
y = combined_data[['G', 'C', 'B', 'A']]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [12]:
# Step 5: Compute Class Weights
y_train_single = np.argmax(y_train.values, axis=1)
y_test_single = np.argmax(y_test.values, axis=1)

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_single), y=y_train_single)


In [13]:
# Step 6: Train the CatBoost Model
cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    class_weights=class_weights.tolist(),
    random_state=42,
    verbose=100
)

print("Starting Model Training...")
cat_model.fit(X_train, y_train_single, eval_set=(X_test, y_test_single), plot=True)


Starting Model Training...


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.5288431	test: 0.5457949	best: 0.5457949 (0)	total: 62ms	remaining: 1m 1s
100:	learn: 0.6473113	test: 0.6508060	best: 0.6512599 (96)	total: 1.3s	remaining: 11.6s
200:	learn: 0.6626656	test: 0.6542097	best: 0.6565691 (166)	total: 2.54s	remaining: 10.1s
300:	learn: 0.6700116	test: 0.6331413	best: 0.6565691 (166)	total: 3.8s	remaining: 8.82s
400:	learn: 0.6744618	test: 0.6244189	best: 0.6565691 (166)	total: 5.79s	remaining: 8.65s
500:	learn: 0.6794320	test: 0.6210507	best: 0.6565691 (166)	total: 7.56s	remaining: 7.53s
600:	learn: 0.6837761	test: 0.6149454	best: 0.6565691 (166)	total: 8.87s	remaining: 5.89s
700:	learn: 0.6866221	test: 0.6107462	best: 0.6565691 (166)	total: 10.2s	remaining: 4.33s
800:	learn: 0.6895830	test: 0.6053986	best: 0.6565691 (166)	total: 11.5s	remaining: 2.87s
900:	learn: 0.6911211	test: 0.6033901	best: 0.6565691 (166)	total: 12.9s	remaining: 1.41s
999:	learn: 0.6928301	test: 0.6028566	best: 0.6565691 (166)	total: 14.1s	remaining: 0us

bestTest = 0.656569

In [14]:
# Step 7: Evaluate the Model
y_pred = cat_model.predict(X_test)
accuracy = accuracy_score(y_test_single, y_pred)
print(f"Updated CatBoost Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_single, y_pred))


Updated CatBoost Model Accuracy: 68.24%

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.67      0.80      2440
           1       0.35      0.54      0.42       416
           3       0.51      0.75      0.61      1117

    accuracy                           0.68      3973
   macro avg       0.61      0.66      0.61      3973
weighted avg       0.79      0.68      0.71      3973



In [15]:
# Step 8: Save the Model
cat_model.save_model('catboost_fault_detection_model_single_phase.cbm')
print("Updated Model saved successfully!")


Updated Model saved successfully!


In [16]:
# Step 9: Feature Importance
print("\nFeature Importance:")
feature_importances = cat_model.get_feature_importance()
for feature, importance in zip(X.columns, feature_importances):
    print(f"{feature}: {importance:.2f}")



Feature Importance:
Ia: 37.13
Va: 44.21
fault_distance: 18.67


In [17]:
# Step 10: Download the Model (Optional)
files.download('catboost_fault_detection_model_single_phase.cbm')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>